In [1]:
from __future__ import annotations

In [2]:
%reload_ext autoreload
%autoreload 3

In [125]:

import numpy as np

from minitorch.tensor.tensor import Tensor
from minitorch.activations.activations import GELU, ReLU, Tanh
from minitorch.nn.layers import Linear, Layer, Module, Parameter
from minitorch.attention.attention import MultiHeadAttention
from minitorch.embendding.embed import EmbeddingLayer
from minitorch.optimizers.optim import SGD, AdamW, Adam
from minitorch.losses.losses import MSE, BCEWithLogits

def create_causal_mask(seq_len: int) -> Tensor:
    """
    Create a causal mask (autoregressive mask).
    
    This create the causal mask to make sure that tokens i
    only communicates to token j where j<i.
    Essential for autoregressive GPT models.

    Args:
        seq_len (int): Length of the sequence

    Returns:
        Tensor: Tensor of shape (1, seq_len, seq_len) with:
        - 1.0 for positions that CAN be attended to (lower triangle)
        - 0.0 for positions that CANNOT be attended to (upper triangle)
    """
    mask = np.tril(np.ones(shape=(seq_len, seq_len), dtype= np.float32))
    return Tensor(mask[np.newaxis, :, :])

In [122]:
xs = np.array([
    [2.0, 3.0, -1.0, 3.0, -1.0, 0.5, 0.7, 0.8, 0.9],
    [3.0, -1.0, 0.5, 0.1, 0.2, 0.3, 0.1, 0.2, 0.3],
    [0.5, 1.0, 1.0, 0.1, 0.2, 0.3, 0.5, 1.0, 1.0],
    [0.1, 0.2, 0.3, 0.5, 1.0, 1.0, 0.5, 1.0, 1.0],
    [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 2.0, 3.0, -1.0],
    [0.7, 0.8, 0.9, 2.0, 3.0, -1.0, 3.0,2.0, 3.0]])

ys = np.array([1, 0, 0, 1, 0, 1])
x_norm = (xs - xs.min(axis=0)) / (xs.max(axis=0)  - xs.min(axis=0) + 1e-8)
x = Tensor(x_norm, dtype=np.float32, requires_grad=True)
y = Tensor(np.array(ys), requires_grad=True)
x

Tensor(data=[[0.65517241 1.         0.         1.         0.         0.75
  0.20689655 0.21428571 0.475     ]
 [1.         0.         0.75       0.         0.3        0.65
  0.         0.         0.325     ]
 [0.13793103 0.5        1.         0.         0.3        0.65
  0.13793103 0.28571428 0.5       ]
 [0.         0.3        0.65       0.13793103 0.5        1.
  0.13793103 0.28571428 0.5       ]
 [0.10344828 0.375      0.8        0.20689655 0.45       0.95
  0.65517241 1.         0.        ]
 [0.20689655 0.45       0.95       0.65517241 1.         0.
  1.         0.64285714 1.        ]], shape=(6, 9), grad_info= True)

In [123]:
from typing import Any


class MLP:
    def __init__(self, nin: int, nout: int, bias=False) -> None:
        self.linear1 = Linear(nin, nout,bias)
        self.gelu_act = ReLU()
        self.linear2 = Linear(nout, 1, bias)
        
    def __call__(self, inputs: Tensor) -> Any:
        out = self.linear1(inputs)
        act = self.gelu_act(out)
        return self.linear2(act)
    
    def parameters(self)-> list[Tensor]:
        all_params= []
        for layer in [self.linear1, self.linear2]:
            all_params.extend(layer.parameters())
        return all_params

In [126]:
in_features = x.shape[-1]
out_features = 5
max_iters = 10000
eva_steps = max_iters // (max_iters / 100)

#* instantiate the model
mlp = MLP(in_features, out_features, bias=True)
optimizer = Adam(mlp.parameters(), lr=0.05)
loss_fn = BCEWithLogits()

for i in range(max_iters):
    #* forward pass
    out = mlp(x)
    optimizer.zero_grad()

    # # #* calculate the loss
    loss = loss_fn(out, y)
    # #* zero grad the grad and do a backward pass
    loss.backward()

    # #* update the parameters
    optimizer.step()
        
    if i % eva_steps == 0:
        print(f'Epoch: {i}, Epoch Loss: {loss.data}')

Epoch: 0, Epoch Loss: 27.132302848307074
Epoch: 100, Epoch Loss: 24.953305826009466
Epoch: 200, Epoch Loss: 24.953298500168575
Epoch: 300, Epoch Loss: 24.95329850015803
Epoch: 400, Epoch Loss: 24.95329850015803
Epoch: 500, Epoch Loss: 24.95329850015803
Epoch: 600, Epoch Loss: 24.95329850015803
Epoch: 700, Epoch Loss: 24.953298500158024
Epoch: 800, Epoch Loss: 24.953298500158024
Epoch: 900, Epoch Loss: 24.953298500158024
Epoch: 1000, Epoch Loss: 24.953298500158024
Epoch: 1100, Epoch Loss: 24.953298500158024
Epoch: 1200, Epoch Loss: 24.953298500158024
Epoch: 1300, Epoch Loss: 24.953298500158024
Epoch: 1400, Epoch Loss: 24.953298500158024
Epoch: 1500, Epoch Loss: 24.953298500158024
Epoch: 1600, Epoch Loss: 24.953298500158024
Epoch: 1700, Epoch Loss: 24.953298500158024
Epoch: 1800, Epoch Loss: 24.953298500158024
Epoch: 1900, Epoch Loss: 24.953298500158024
Epoch: 2000, Epoch Loss: 24.953298500158024
Epoch: 2100, Epoch Loss: 24.953298500158024
Epoch: 2200, Epoch Loss: 24.953298500158024
Epoc

In [121]:
out = mlp(x)
out

Tensor(data=[[ 1.38777878e-17]
 [-3.05311332e-16]
 [ 1.38777878e-16]
 [-2.91433544e-16]
 [ 1.24900090e-16]
 [-8.32667268e-17]], shape=(6, 1), grad_info= True)